In [18]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm


# Config of Model Training

In [19]:
EPOCHS = 100
# BATCH_SIZE = 128
LR = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
print(f"Using device: {DEVICE}")


Using device: cuda


# Loading in training and test datasets for CIFAR 100

In [20]:
# Data Loaders
cifar100_mean, cifar100_std = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761) # for CIFAR100 dataset
DATA_TRANSFORM = transforms.Compose([
                          transforms.RandomCrop(32, padding=4),
                          transforms.RandomHorizontalFlip(),
                          transforms.ToTensor(),
                          transforms.Normalize(cifar100_mean, cifar100_std)])

full_data = datasets.CIFAR100(root='./data', train=True, download=True, transform=DATA_TRANSFORM)

train_split = int(0.8*len(full_data))
val_split = len(full_data) - train_split

train_data, val_data = torch.utils.data.random_split(full_data, [train_split, val_split],
                                                     generator=torch.Generator().manual_seed(SEED))
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False)

# batch_train, batch_val = 64, 256
# train_loader = DataLoader(
#     datasets.CIFAR100("data",
#                       train=True,
#                       download=True,
#                       transform=transforms.Compose([
#                           transforms.RandomCrop(32, padding=4),
#                           transforms.RandomHorizontalFlip(),
#                           transforms.ToTensor(),
#                           transforms.Normalize(cifar100_mean, cifar100_std)])),
#     batch_size=batch_train, shuffle=True)

# test_loader = DataLoader(
#     datasets.CIFAR100("data",
#                       train=False,
#                       download=True,
#                       transform=transforms.Compose([
#                           transforms.ToTensor(),
#                           transforms.Normalize(cifar100_mean, cifar100_std)])),
#     batch_size=batch_val, shuffle=False)


# Import ResNet18 Model

In [24]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False) # adapt for 32x32 format of CIFAR100
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


# Training Metrics and Utilities

In [25]:
@torch.no_grad()
def accuracy(logits: torch.Tensor, y: torch.Tensor) -> float:
    """Compute classification accuracy from logits."""
    return (logits.argmax(1) == y).float().mean().item()


def train_one_epoch(model: nn.Module, loader: DataLoader, criterion, optimizer) -> tuple[float, float]:
    """Standard training loop for one epoch."""
    model.train()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:                      # x: [B, C, H, W]
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)                    # [B, 10]
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        bs = x.size(0)
        tot += bs
        tot_loss   += loss.item() * bs
        tot_correct+= (logits.argmax(1) == y).sum().item()
    return tot_loss / tot, tot_correct / tot


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion) -> tuple[float, float]:
    """Evaluation loop."""
    model.eval()
    tot_loss, tot_correct, tot = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        bs = x.size(0)
        tot += bs
        tot_loss   += loss.item() * bs
        tot_correct+= (logits.argmax(1) == y).sum().item()
    return tot_loss / tot, tot_correct / tot

def plot_history(history: dict, title: str):
    """Plot training history."""
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'],   label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend()
    plt.subplot(1,2,2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'],   label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend()
    plt.suptitle(title)
    plt.tight_layout(); plt.show()


@torch.no_grad()
def show_errors(model: nn.Module, loader: DataLoader, n: int = 6):
    """Display misclassified examples."""
    model.eval(); shown = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(1)
        wrong = torch.where(preds != y)[0]
        for i in wrong[:n]:
            # Check if the image is 1-channel or 3-channel
            img_data = x[i].cpu().squeeze()
            if img_data.ndim == 2: # Grayscale image
                plt.imshow(img_data, cmap='gray')
            else: # Color image (e.g., from transfer learning transform)
                # Permute dimensions from (C, H, W) to (H, W, C) for matplotlib
                plt.imshow(img_data.permute(1, 2, 0))

            # plt.title(f"True: {CLASSES[y[i].item()]} | Pred: {CLASSES[preds[i].item()]}")
            plt.axis('off'); plt.show(); shown += 1
        if shown >= n: break

# Training Run

In [26]:
print(f"{'Epoch':>6} {'Loss':>8} {'Val Top-1':>10} {'Val Top-5':>10}")
print("-" * 40)

def train_one_epoch(model, loader, criterion, optimizer):
    """Standard training loop"""
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        total += batch_size
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == y).sum().item()

    return total_loss / total, total_correct / total



train_accs, val_accs = [], []
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"  Epoch {epoch+1}/{EPOCHS} | Train: {train_acc:.3f} | Val: {val_acc:.3f}")


torch.save(model.state_dict(), "resnet18_cifar100.pth")
print("\nDone. Model saved to resnet18_cifar100.pth")

 Epoch     Loss  Val Top-1  Val Top-5
----------------------------------------
  Epoch 1/100 | Train: 0.362 | Val: 0.525
  Epoch 2/100 | Train: 0.611 | Val: 0.609
  Epoch 3/100 | Train: 0.692 | Val: 0.647
  Epoch 4/100 | Train: 0.739 | Val: 0.659
  Epoch 5/100 | Train: 0.780 | Val: 0.669
  Epoch 6/100 | Train: 0.808 | Val: 0.679
  Epoch 7/100 | Train: 0.831 | Val: 0.684
  Epoch 8/100 | Train: 0.852 | Val: 0.705
  Epoch 9/100 | Train: 0.872 | Val: 0.710
  Epoch 10/100 | Train: 0.889 | Val: 0.705
  Epoch 11/100 | Train: 0.901 | Val: 0.694
  Epoch 12/100 | Train: 0.911 | Val: 0.698
  Epoch 13/100 | Train: 0.922 | Val: 0.720
  Epoch 14/100 | Train: 0.930 | Val: 0.713
  Epoch 15/100 | Train: 0.938 | Val: 0.712
  Epoch 16/100 | Train: 0.947 | Val: 0.723
  Epoch 17/100 | Train: 0.950 | Val: 0.713
  Epoch 18/100 | Train: 0.953 | Val: 0.724
  Epoch 19/100 | Train: 0.958 | Val: 0.717
  Epoch 20/100 | Train: 0.962 | Val: 0.730
  Epoch 21/100 | Train: 0.965 | Val: 0.710
  Epoch 22/100 | Train: 0.9